# Use cohorts for machine learning

This guide goes from a cohort to training-ready data: implement a small `TaskAdapter`, pull one concatenated
`AnnData` table out of several stores without downloading images, and loop over the train fold of a
study-level split. It uses four small stores (4 to 10 MB each):

- [`xVK6T0rajFTgOKci0000`](https://spatialdata-db-web.pages.dev/datasets/visium-hd-bone-element-xvk6t0rajftgokci0000/): VisiumHD, bone element
- [`cgmLrEw3hgv7yhZf0000`](https://spatialdata-db-web.pages.dev/datasets/visium-spatial-gene-expression-head-of-femur-cgmlrew3hgv7yhzf0000/): Visium, head of femur
- [`h1okQyQN4GJd0I1l0000`](https://spatialdata-db-web.pages.dev/datasets/visium-spatial-gene-expression-trunk-h1okqyqn4gjd0i1l0000/): Visium, mouse trunk
- [`EMS0unVubRENrRu80000`](https://spatialdata-db-web.pages.dev/datasets/visium-spatial-gene-expression-trunk-ems0unvubrenrru80000/): Visium, mouse trunk

```
pip install git+https://github.com/theislab/spatialdata-db-client
```


In [1]:
import sddb

cat = sddb.Catalog()
cohort = cat.cohort(uids=["xVK6T0rajFTgOKci0000", "cgmLrEw3hgv7yhZf0000", "h1okQyQN4GJd0I1l0000", "EMS0unVubRENrRu80000"])
cohort

<SpatialDataCohort: 4 datasets>

## Pin the catalog version

A dated release is pinned: `Catalog(version="2026-10-28")` always reads that release, not whatever is current, and `verify()` raises when the catalog file has changed since a cohort was frozen.

In [ ]:
cat = sddb.Catalog(version="2026-10-28")

## One table across many stores

`to_anndata(table=...)` reads only each member's `tables/<name>` group (no images, no shapes) and concatenates
them with an outer join. `obs["uid"]` says which dataset a row came from and `obs["study_id"]` which study.

In [2]:
visium = cohort.filter(technology="Visium")
adata = visium.to_anndata(table="table")
print(adata)
adata.obs["uid"].value_counts().reindex([d.uid for d in visium])

AnnData object with n_obs × n_vars = 5362 × 71642
    obs: 'in_tissue', 'array_row', 'array_col', 'spot_id', 'region', 'uid', 'study_id'
    obsm: 'spatial'
    layers: 'csc_counts'


uid
EMS0unVubRENrRu80000     192
cgmLrEw3hgv7yhZf0000    4992
h1okQyQN4GJd0I1l0000     178
Name: count, dtype: int64

## Mix sources with gene_key

HEST keys genes by Ensembl id and 10x by symbol, so without `gene_key` a mixed cohort shares no genes. For `qEP9pd4LlyKeufe20000` and `USx288NLowiqKbV30000` the two tables share 0 genes by default and 17756 genes with `gene_key="ensembl"`. Use `gene_key="symbol"` to align on symbols instead.

In [ ]:
mixed = cat.cohort(uids=["qEP9pd4LlyKeufe20000", "USx288NLowiqKbV30000"])
aligned = mixed.to_anndata(gene_key="ensembl")

## Open stores as SpatialData

`to_list()` opens every member as `SpatialData` (`to_dict()` keys them by uid). Stores are fetched in parallel
into the local cache and reused on the next call; `elements=[...]` copies only the listed elements. Pass
`dest="data/"` to keep the full stores in a folder of your choice, laid out like `download()`.

In [3]:
sdatas = visium.to_dict(elements=["tables/table"])
{uid: s.tables["table"].shape for uid, s in sdatas.items()}

{'EMS0unVubRENrRu80000': (192, 53574),
 'cgmLrEw3hgv7yhZf0000': (4992, 18085),
 'h1okQyQN4GJd0I1l0000': (178, 53574)}

In [4]:
full = visium[:1].to_list()[0]
print("images:", len(full.images), "| tables:", list(full.tables))

images: 1 | tables: ['table']


## A minimal TaskAdapter

`TaskAdapter` is a protocol: the client defines the contract and your task repository implements it. An adapter
declares its `name`, `version` and `config`, states its `requirements()`, and provides three hooks:

- `validate_meta(meta)`: cheap, metadata-only check (advisory), called by the task framework with an
  `ObjectMeta` (the element inventory, read without any array chunk).
- `validate(sdata)`: authoritative check on the opened object.
- `build(sdata)`: produce the task item for one object.

Each hook returns a list of issue strings, empty when all is well (`build` returns the item).

In [5]:
from typing import Any

from sddb import TaskAdapter


class SpotSummaryAdapter:
    """Per-slide summary: number of spots, genes and total counts."""

    name = "spot-summary"
    version = "0.1.0"

    def __init__(self, min_spots: int = 1_000) -> None:
        self.config: dict[str, Any] = {"min_spots": min_spots}

    def requirements(self) -> dict[str, Any]:
        return {"tables": 1, "elements": ["tables/*"]}

    def validate_meta(self, meta) -> list[str]:
        return [] if meta.tables else ["no table in store"]

    def validate(self, sdata) -> list[str]:
        table = next(iter(sdata.tables.values()))
        n = self.config["min_spots"]
        return [] if table.n_obs >= n else [f"only {table.n_obs} spots (< {n})"]

    def build(self, sdata) -> dict[str, float]:
        table = next(iter(sdata.tables.values()))
        return {"n_spots": table.n_obs, "n_genes": table.n_vars, "total_counts": float(table.X.sum())}


adapter = SpotSummaryAdapter()
isinstance(adapter, TaskAdapter)

True

## Loop over a split's train fold

`split()` is group-safe: every study lands in exactly one fold. Datasets without a `study_id` form their own
group. Each fold is itself a cohort, so you can iterate it, check each object with the adapter and build items. Members keep catalog order throughout this guide, and outputs are keyed by uid.

In [6]:
folds = cohort.split(by="study_id", train=0.5, val=0.25, test=0.25, seed=0)
{name: [d.uid for d in fold] for name, fold in folds.items()}

{'train': ['EMS0unVubRENrRu80000', 'cgmLrEw3hgv7yhZf0000'],
 'val': ['xVK6T0rajFTgOKci0000'],
 'test': ['h1okQyQN4GJd0I1l0000']}

In [7]:
items = {}
for ds in folds["train"]:
    table = ds.default_table or "table"
    sdata = ds.to_sdata(elements=[f"tables/{table}"])
    issues = adapter.validate(sdata)
    if issues:
        print(ds.uid, "skipped:", issues)
        continue
    items[ds.uid] = adapter.build(sdata)
items

EMS0unVubRENrRu80000 skipped: ['only 192 spots (< 1000)']


{'cgmLrEw3hgv7yhZf0000': {'n_spots': 4992,
  'n_genes': 18085,
  'total_counts': 1069313.0}}

Because the split manifest travels with the fold, you can store it next to your results and recover the
exact assignment later (see the cohorts guide).

In [8]:
folds["train"].split_manifest.assignments

{'EMS0unVubRENrRu80000': 'train',
 'cgmLrEw3hgv7yhZf0000': 'train',
 'xVK6T0rajFTgOKci0000': 'val',
 'h1okQyQN4GJd0I1l0000': 'test'}

In [9]:
print("Catalog generated at:", cat.generated_at)

Catalog generated at: 2026-10-26
